# SCRUM-53 - train yolov8n-cls on the symbol-detector dataset

Run this on a free GPU notebook (Kaggle or Colab - Kaggle: Settings > Accelerator > GPU T4 x2 or P100; Colab: Runtime > Change runtime type > GPU).

Clones `symbol-detector` at `feature/scrum-51-image-dataset`, rebuilds the
YOLO classification folder tree from `data/manifest.csv`
(`prepare_yolo_dataset.py`), trains `yolov8n-cls`, evaluates on the
held-out test split, and exports to ONNX.

Classes: the 12 banned symbols, `swastika_religious` (must be reported as
`needs_context`, never `banned` - SCRUM-53), and `none`.

In [ ]:
!git clone --branch feature/scrum-51-image-dataset https://github.com/LegalAidsocial/symbol-detector.git
%cd symbol-detector

In [ ]:
!pip install -q -e . ultralytics onnx onnxruntime

In [ ]:
!python -m symbol_detector.prepare_yolo_dataset

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")
results = model.train(
    data="yolo_dataset",
    epochs=60,
    imgsz=224,
    patience=15,
    project="runs_scrum53",
    name="yolov8n_cls_symbols",
    seed=20261109,
)

## Evaluate on the held-out test split

`yolo_dataset/test` was never used for training or validation - this is
the real check, not the training curves above.

In [ ]:
best = YOLO("runs_scrum53/yolov8n_cls_symbols/weights/best.pt")
test_metrics = best.val(data="yolo_dataset", split="test")
print("top1 accuracy on locked test split:", test_metrics.top1)
print("top5 accuracy on locked test split:", test_metrics.top5)

## Export to ONNX for CPU inference

In [ ]:
onnx_path = best.export(format="onnx", imgsz=224, opset=12, simplify=True)
print("exported:", onnx_path)

# Class name order matters for decoding ONNX Runtime output on the CPU side -
# save it alongside the model.
import json
names = best.names  # {0: 'black_sun', 1: 'celtic_cross', ...}
with open("symbol_classifier.names.json", "w") as f:
    json.dump(names, f, indent=2)
print(names)

## Download these two files and put them in the repo's `models/` folder:
- `runs_scrum53/yolov8n_cls_symbols/weights/best.onnx`
- `symbol_classifier.names.json`

In [ ]:
# Kaggle: files land in /kaggle/working/ - download from the Output tab.
# Colab: uncomment to trigger a browser download directly.
# from google.colab import files
# files.download("runs_scrum53/yolov8n_cls_symbols/weights/best.onnx")
# files.download("symbol_classifier.names.json")